In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

import os

load_dotenv()

llm = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
)

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50,
)

loader = TextLoader("./genshinImpact.txt")
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings(
    base_url=os.getenv("OPENAI_EMBEDDING_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_EMBEDDING_MODEL"),
)

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up. \n{context}"),
    ("human", "{question}"),
])

stuff_chain = (
    {
    "context": retriever,
    "question": RunnablePassthrough(),
    } | prompt | llm
)

In [ ]:
map_doc_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up. \n{context}"),
        ("human", "{question}"),
    ]
)

map_doc_chain = map_doc_prompt | llm


def map_docs(inputs):
    documents = inputs["documents"]
    question = inputs["question"]
    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": doc.page_content, "question": question}
        ).content
        for doc in documents
    )


map_chain = {
    "documents": retriever,
    "question": RunnablePassthrough(),
} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up. \n{context}"),
        ("human", "{question}"),
    ]
)

chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

In Genshin Impact, **Archons** are the powerful elemental gods who rule over each major region of the world, Teyvat.

- Each nation (like Mondstadt, Liyue, Inazuma) has its own Archon.
- Every Archon is tied to:
  - a specific **element** (such as Anemo, Geo, Electro), and  
  - a guiding **ideal** or concept (like freedom, contracts, or eternity).  

They act as the protector and symbolic ruler of their region, and the Traveler often meets and deals with these Archons during the story.


In [5]:
from langchain_core.callbacks import get_usage_metadata_callback

with get_usage_metadata_callback() as st:
    stuff_result = stuff_chain.invoke("What does Archons mean?")

print("=== Stuff Chain ===")
print(stuff_result.content)
print(st.usage_metadata)

with get_usage_metadata_callback() as mp:
    map_result = chain.invoke("What does Archons mean?")

print("=== Map Reduce Chain ===")
print(map_result.content)
print(mp.usage_metadata)

=== Stuff Chain ===
In the context of Genshin Impact, **Archons** are the gods who rule over each region of Teyvat.

- Each major region (like Mondstadt, Liyue, Inazuma) has an Archon.
- They are powerful beings tied to a specific **element** (like Anemo, Geo, Electro) and to a **theme** (freedom, contracts, eternity, etc.).
- The story gradually reveals more about these Archons and their role in the world and its history.
{'gpt-5.1-2025-11-13': {'input_tokens': 528, 'output_tokens': 115, 'total_tokens': 643, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
=== Map Reduce Chain ===
In Genshin Impact, **Archons** are the **elemental gods who rule over each major region of Teyvat**.

- Each big nation (like Mondstadt, Liyue, Inazuma) has its own Archon.
- Every Archon is tied to a specific **element** (such as Anemo, Geo, Electro).
- Each also represents a **theme** (for example: freedom, contracts, eternity).
- They are extreme